<div dir="rtl">

# الدرس 28: التحقق المتقاطع

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### الطيات بعينك

</div>

In [ ]:
import numpy as np
from sklearn.model_selection import KFold

X10 = np.arange(10).reshape(-1, 1)
for i, (tr, va) in enumerate(KFold(n_splits=5).split(X10), start=1):
    print(f"round {i}: validate on {va}, train on {tr}")

<div dir="rtl">

### التحقق المتقاطع لخط معالجة

</div>

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=cv)
print("fold scores:", scores.round(3))
print(f"mean = {scores.mean():.3f}, std = {scores.std():.3f}")

<div dir="rtl">

### كم يقفز التقسيم الواحد؟

</div>

In [ ]:
from sklearn.model_selection import train_test_split

single = []
for seed in range(20):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    single.append(model.fit(X_tr, y_tr).score(X_te, y_te))
single = np.array(single)
print(f"single splits: min = {single.min():.3f}, max = {single.max():.3f}")

<div dir="rtl">

### مقارنة نماذج بنفس الطيات

</div>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "knn":      make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "forest":   RandomForestClassifier(n_estimators=200, random_state=0),
}
for name, m in models.items():
    s = cross_val_score(m, X, y, cv=cv)
    print(f"{name:9} {s.mean():.3f} ± {s.std():.3f}")

<div dir="rtl">

### الطيات حسب المجموعات والوقت

</div>

In [ ]:
from sklearn.model_selection import GroupKFold, TimeSeriesSplit

groups = np.array([0, 0, 1, 1, 1, 2, 2, 3, 3, 3])     # مثلاً: رقم المريض لكل صورة
for tr, va in GroupKFold(n_splits=4).split(X10, groups=groups):
    print("validate groups", np.unique(groups[va]), "rows", va)

print()
for tr, va in TimeSeriesSplit(n_splits=4).split(X10):
    print("train", tr, "-> validate", va)

<div dir="rtl">

## تمرين: قارن بعدل

استخدم بيانات النبيذ من الدرس 15.

1. قارن بالتحقق المتقاطع الطبقي بعشر طيات بين: أقرب الجيران بدون توحيد، وأقرب الجيران مع التوحيد في خط معالجة.
2. اطبع المتوسط والانحراف لكل واحد.
3. هل الفرق حقيقي، أم داخل التذبذب؟
4. سؤال: لماذا يجب أن يكون التوحيد داخل خط المعالجة هنا، وليس قبل التحقق المتقاطع؟

</div>

In [ ]:
from sklearn.datasets import load_wine

Xw, yw = load_wine(return_X_y=True)
# قارن النموذجين بنفس الطيات